# 01 — Data Extraction

Detects extreme temperature events for each ensemble member and extracts:
- Per-event area-averaged temperature time series (factual + counterfactual)
- Event barycentres (lat, lon)
- Full global SLP anomaly field `(T, n_lat, n_lon)` — **not pre-sliced** so
  attribution methods can freely choose their spatial domain at run time.

**Edit the CONFIG block below, then Run All.**

In [1]:
import sys, os

# ── Update this path to point to the src/ folder ──
SRC_PATH = '/home/homer/Documents/Research/Projects/ELLIS_winter2026/Challenge_Attribution_ML/src'
sys.path.insert(0, SRC_PATH)

from src.data_utils import (
    detect_extreme_events,
    extract_event_fast,
    event_frequency_map,
    get_smoothed_gmt,
)
import numpy as np
import xarray as xr
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed

/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ================================================================
#  CONFIG  —  edit here
# ================================================================
data_path = '/home/homer/Documents/Research/Data/attribution_evaluation_data/'
CONFIG = {
    # Variable names in the NetCDF files
    'var':        'tasmax',
    'var_slp':    'psl',

    # Event detection
    'start_year': 2004,
    'percentile': 99,   # climatological percentile threshold
    'min_area':   30,     # minimum connected grid-cells per event

    # Ensemble
    'n_members':  10,
    'n_jobs':     4,      # parallel workers (-1 = all CPUs)

    # Input NetCDF paths  (update to your Drive location)
    'path_tf': data_path + 'full_data/tasmax_historical_CMIP.IPSL.IPSL-CM6A-LR.historical.Amon.gr_1850-2014.nc',
    'path_tc': data_path + 'full_data/tasmax_hist-nat_DAMIP.IPSL.IPSL-CM6A-LR.hist-nat.Amon.gr_1850-2014.nc',
    'path_sc': data_path + 'full_data/psl_hist-nat_DAMIP.IPSL.IPSL-CM6A-LR.hist-nat.Amon.gr_1850-2014.nc',
    'path_sf': data_path + 'full_data/psl_historical_CMIP.IPSL.IPSL-CM6A-LR.historical.Amon.gr_1850-2014.nc',

    # Output folder
    'out_dir':    data_path + 'extracted/',
}
# ================================================================

In [3]:
def process_member(m):
    with (
        xr.open_dataset(CONFIG['path_tf']).sel(member_id=m).squeeze() as run_tf,
        xr.open_dataset(CONFIG['path_tc']).sel(member_id=m).squeeze() as run_tc,
        xr.open_dataset(CONFIG['path_sf']).sel(member_id=m).squeeze() as run_sf,
        xr.open_dataset(CONFIG['path_sc']).sel(member_id=m).squeeze() as run_sc,
    ):
        # Monthly anomalies — remove seasonal cycle
        run_tf_anom = run_tf.groupby('time.month') - run_tf.groupby('time.month').mean()
        run_tc_anom = run_tc.groupby('time.month') - run_tc.groupby('time.month').mean()
        run_sf_anom = run_sf.groupby('time.month') - run_sf.groupby('time.month').mean()
        run_sc_anom = run_sc.groupby('time.month') - run_sc.groupby('time.month').mean()

        # Remove global mean from SLP (retain spatial patterns only)
        run_sf_anom = run_sf_anom - run_sf_anom.mean(dim=['lat', 'lon'])
        run_sc_anom = run_sc_anom - run_sc_anom.mean(dim=['lat', 'lon'])

        # Detect events on factual run; apply same mask to counterfactual
        mask     = detect_extreme_events(
            run_tf_anom, CONFIG['var'], CONFIG['percentile'], CONFIG['min_area'])
        freq_map = event_frequency_map(mask)

        f_tas, f_tas_v, idx_f, event_loc = extract_event_fast(
            run_tf_anom, mask, CONFIG['var'], CONFIG['start_year'])
        c_tas, c_tas_v, idx_c, _ = extract_event_fast(
            run_tc_anom, mask, CONFIG['var'], CONFIG['start_year'])

        if f_tas is None:
            print(f'[{m}] No events found — skipping.')
            return None

        slp_da_f = run_sf_anom[CONFIG['var_slp']]  # (T, lat, lon)
        slp_da_c = run_sc_anom[CONFIG['var_slp']]

        gmt4_f = get_smoothed_gmt(
            run_tf_anom[CONFIG['var']].mean(dim=['lat', 'lon']).values)
        gmt4_c = get_smoothed_gmt(
            run_tc_anom[CONFIG['var']].mean(dim=['lat', 'lon']).values)


        return {
            'member':              m,
            'times':               run_tf_anom.time.values,
            'gmt4_f':              gmt4_f,             # (T,)
            'gmt4_c':              gmt4_c,
            'location':            event_loc,           # (n_events, 2) — [lat, lon]
            'slp_lat':             slp_da_f.lat.values, # (n_lat,)
            'slp_lon':             slp_da_f.lon.values, # (n_lon,)
            'idx_f':               idx_f,               # (n_events,)
            'idx_c':               idx_c,
            'f_tas':               f_tas,               # (T, n_events)
            'f_tas_vals':          f_tas_v,             # (n_events,)
            'c_tas':               c_tas,
            'c_tas_vals':          c_tas_v,
            'f_slp':               slp_da_f.values,     # (T, n_lat, n_lon)
            'c_slp':               slp_da_c.values,
            'event_frequency_map': freq_map,
        }

In [4]:
os.makedirs(CONFIG['out_dir'], exist_ok=True)
members = [f'r{i}i1p1f1' for i in range(1, CONFIG['n_members'] + 1)]
print(f'Processing {len(members)} members with {CONFIG["n_jobs"]} workers...')

results = Parallel(n_jobs=CONFIG['n_jobs'])(
    delayed(process_member)(m) for m in tqdm(members)
)
results = [r for r in results if r is not None]
print(f'Done — {len(results)} members extracted.')

Processing 10 members with 4 workers...


  0%|          | 0/10 [00:00<?, ?it/s]/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
100%|██████████| 10/10 [05:22<00:00, 32.30s/it]
/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._uredu

Done — 10 members extracted.


In [5]:
out_name = (
    f"extracted_{CONFIG['var']}"
    f"_nmem{len(results)}"
    f"_start{CONFIG['start_year']}"
    f"_p{CONFIG['percentile']}.pkl"
)
out_path = os.path.join(CONFIG['out_dir'], out_name)

with open(out_path, 'wb') as fh:
    pickle.dump(results, fh)

print(f'Saved -> {out_path}')

Saved -> /home/homer/Documents/Research/Data/attribution_evaluation_data/extracted/extracted_tasmax_nmem10_start2004_p99.9.pkl
